# Track eta-phi maps

Reads and plots a 2D tracking monitoring element (by default `TrackEtaPhi_ImpactPoint_GenTk`, the number of high-purity tracks -- pt > 1 GeV -- as a function of eta and phi at the track's impact point) from DIALS.

Supports plotting individual lumisections, and also comparing a single "target" lumisection side by side with a "reference" built by summing a range of lumisections (or a full run) -- e.g. to compare a lumisection suspected of an anomaly against the run's typical behaviour -- plus a third panel with the ratio of the two after normalizing each to a common scale.

Note: this tracking monitoring element lives in the `ZeroBias`/`PromptReco` dataset -- the same one used for the pixel cluster occupancy MEs elsewhere in this repo, and the dataset the original (manual) version of these plots was made with. It was found to only start being populated at run 379413 (an earlier candidate, `StreamExpress`, was tried first but abandoned for consistency with the original studies) -- runs before that have no data for this ME in either stream.

In [ ]:
import os
import re
import json
import numpy as np
import matplotlib.pyplot as plt

from cmsdials import Dials
from cmsdials.auth.bearer import Credentials
from cmsdials.filters import LumisectionHistogram2DFilters

In [ ]:
# authenticate; on first use this will print a device-login URL to visit.
# credentials are cached in .cache/ next to this notebook, so subsequent
# runs (within the token's validity) won't need to re-authenticate.
creds = Credentials.from_creds_file(cache_dir='.cache')
dials = Dials(creds, workspace='tracker')

In [ ]:
# --- configuration ---

# the monitoring element to plot.
# a few other candidates, in case this one turns out not to be the right one
# (see the notebook that identified it for how these were found):
#   Tracking/TrackParameters/highPurityTracks/pt_1/GeneralProperties/TrackEta_ImpactPoint_GenTk   (1D, eta only)
#   Tracking/TrackParameters/highPurityTracks/pt_1/GeneralProperties/TrackPhi_ImpactPoint_GenTk   (1D, phi only)
#   Tracking/TrackParameters/generalTracks/GeneralProperties/TrackEtaHighpurity_ImpactPoint_GenTk (1D, no explicit pt cut)
ME = 'Tracking/TrackParameters/highPurityTracks/pt_1/GeneralProperties/TrackEtaPhi_ImpactPoint_GenTk'

# dataset to query; a regex, since the exact processing version (v1, v2, ...)
# can differ per era. ZeroBias/PromptReco -- same dataset as the pixel cluster
# occupancy MEs used elsewhere in this repo, and as the original manual
# version of these plots -- for consistency (StreamExpress also has this ME,
# but isn't used here anymore; see the intro note above).
DATASET_REGEX = 'ZeroBias'

# list of (run, lumisection) pairs to plot individually; edit freely.
RUNS_LS = [
    (381516, 532),
    (381516, 533),
]

# target-vs-reference-vs-ratio comparisons to run by default; each entry is
# (target_run, target_ls, ref_run, ref_ls_min, ref_ls_max) -- ref_run=None
# defaults to target_run, and ref_ls_min=ref_ls_max=None means the reference
# is built from the full run's statistics. Edit/extend freely.
COMPARISONS = [
    (392250, 633, 392250, None, None),
    (396871, 743, 396871, None, None),
]

# output directory for saved plots (see save_trackmap()/save_compare_trackmaps() below).
PLOT_DIR = 'output_test'

# restrict every fetched/loaded track map to this eta (x-axis) range before
# any ratio/significance computation -- tracks with |eta| > 2.5 aren't used
# in later stages anyway, and left in they dominate the "most significant
# bin" pick with low-acceptance edge-bin artifacts (see _clip_eta()). Set to
# None to fall back to DIALS's full eta range. The on-disk reference cache
# (run_distributions_cache/) is deliberately left untouched -- it always
# stores the full range; load_cached_trackmap_sum() clips after loading, so
# it stays reusable across different eta cuts (see cache_reference_trackmaps()
# passing clip_eta_range=None explicitly when it writes the cache).
CLIP_ETA_RANGE = (-2.5, 2.5)

In [ ]:
def _clip_eta(data, x_min, x_max, y_min, y_max, eta_range):
    '''Restricts a 2D track map to the given eta range (the x-axis; None
    leaves it unchanged) by dropping whole columns whose bin center falls
    outside the range, and shrinking x_min/x_max to match the kept columns.
    Used to apply CLIP_ETA_RANGE uniformly wherever a track map is fetched
    or loaded, rather than baking it into what's stored on disk -- see
    load_cached_trackmap_sum() (clips after np.load()) and
    cache_reference_trackmaps() (passes clip_eta_range=None so the cached
    file itself always keeps the full range). Returns (clipped_data,
    new_x_min, new_x_max, y_min, y_max).'''
    if eta_range is None:
        return data, x_min, x_max, y_min, y_max
    eta_lo, eta_hi = eta_range
    n_cols = data.shape[1]
    bin_width = (x_max - x_min) / n_cols
    centers = x_min + (np.arange(n_cols) + 0.5) * bin_width
    keep = np.where((centers >= eta_lo) & (centers <= eta_hi))[0]
    col_lo, col_hi = keep[0], keep[-1] + 1
    return data[:, col_lo:col_hi], x_min + col_lo * bin_width, x_min + col_hi * bin_width, y_min, y_max


def get_trackmap(run, ls, me=ME, dataset_regex=DATASET_REGEX, clip_eta_range=CLIP_ETA_RANGE):
    '''Fetches a single 2D histogram for the given (run, ls), restricted to
    clip_eta_range (see _clip_eta(); defaults to CLIP_ETA_RANGE, pass None
    for the full DIALS range). Returns (data, x_min, x_max, y_min, y_max,
    entries, dataset) or None if not found -- entries is recomputed from the
    (possibly clipped) data rather than taken from DIALS, so it stays
    consistent with what data actually contains.'''
    filters = LumisectionHistogram2DFilters(
        me=me,
        dataset__regex=dataset_regex,
        run_number=run,
        ls_number=ls,
    )
    result = dials.h2d.list(filters)
    if not result.results:
        print(f'WARNING: no data found for run {run}, LS {ls} (me={me}, dataset regex={dataset_regex}).')
        return None
    if len(result.results) > 1:
        datasets = [r.dataset for r in result.results]
        print(f'WARNING: run {run}, LS {ls} matched {len(result.results)} datasets ({datasets}); using the first one.')
    r = result.results[0]
    data = np.array(r.data)
    data, x_min, x_max, y_min, y_max = _clip_eta(data, r.x_min, r.x_max, r.y_min, r.y_max, clip_eta_range)
    return data, x_min, x_max, y_min, y_max, int(data.sum()), r.dataset

In [ ]:
def get_trackmap_sum(run, ls_min=None, ls_max=None, me=ME, dataset_regex=DATASET_REGEX, clip_eta_range=CLIP_ETA_RANGE):
    '''Fetches and sums 2D histograms over lumisections [ls_min, ls_max] in
    a single run (both None = the full run), restricted to clip_eta_range
    (see _clip_eta(); defaults to CLIP_ETA_RANGE, pass None for the full
    DIALS range -- cache_reference_trackmaps() does this explicitly so the
    on-disk reference cache stays full-range). Returns (summed_data, x_min,
    x_max, y_min, y_max, total_entries, n_ls, dataset), or None if nothing
    was found. n_ls is how many lumisections actually contributed (the
    configured dataset stream may not have data for every lumisection, so
    this can be fewer than ls_max-ls_min+1); total_entries is recomputed
    from the (possibly clipped) summed data.'''
    filters = LumisectionHistogram2DFilters(
        me=me,
        dataset__regex=dataset_regex,
        run_number=run,
    )
    result = dials.h2d.list_all(filters, enable_progress=False)
    results = result.results
    if ls_min is not None:
        results = [r for r in results if r.ls_number >= ls_min]
    if ls_max is not None:
        results = [r for r in results if r.ls_number <= ls_max]
    if not results:
        msg = f'WARNING: no data found for run {run}'
        if ls_min is not None or ls_max is not None: msg += f', LS {ls_min}-{ls_max}'
        msg += f' (me={me}, dataset regex={dataset_regex}).'
        print(msg)
        return None
    data = np.sum([np.array(r.data) for r in results], axis=0)
    r0 = results[0]
    data, x_min, x_max, y_min, y_max = _clip_eta(data, r0.x_min, r0.x_max, r0.y_min, r0.y_max, clip_eta_range)
    return data, x_min, x_max, y_min, y_max, int(data.sum()), len(results), r0.dataset

In [ ]:
def normalize_top_half(data):
    '''Normalizes a 2D histogram to a common scale, using only the top 50%
    of bins by value to define that scale -- so a handful of anomalously
    low bins (the very effect we want to characterize) don't skew the
    normalization used to compare it against another histogram. Returns
    (normalized_data, scale).'''
    flat = data.flatten()
    median = np.median(flat)
    top_half = flat[flat >= median]
    scale = np.mean(top_half) if len(top_half) else 1.0
    return (data / scale if scale else data), scale

In [ ]:
def _plot_trackmap_data(data, x_min, x_max, y_min, y_max, ax=None, year=None, run=None, title=None,
                         cmap='viridis', vmin=None, vmax=None, cbar_label='Number of tracks'):
    '''Shared plotting/styling logic (CMS-public-plot style: "CMS Preliminary"
    top-left, "<year> (13.6 TeV)" top-right) for a single already-fetched
    (possibly summed, possibly normalized) 2D histogram. Returns (fig, ax, im).'''
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    else:
        fig = ax.figure

    im = ax.imshow(
        data, origin='lower', aspect='equal',
        extent=[x_min, x_max, y_min, y_max],
        cmap=cmap, vmin=vmin, vmax=vmax,
    )
    # pin the axes limits to exactly the image extent, so neither the
    # header text below nor anything else causes matplotlib to autoscale
    # the axes beyond the actual data -- this is also what keeps the
    # colorbar (tied to the axes height) matching the plot instead of
    # stretching to fill extra whitespace.
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

    ax.set_xlabel('Track $\\eta$', fontsize=14)
    ax.set_ylabel('Track $\\phi$', fontsize=14)
    ax.tick_params(labelsize=12)

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(cbar_label, fontsize=14)
    cbar.ax.tick_params(labelsize=12)

    if year is None and run is not None:
        year = 2024 if run < 390000 else 2025
    ax.text(0.0, 1.01, 'CMS', fontsize=17, fontweight='bold', ha='left', va='bottom', transform=ax.transAxes)
    ax.text(0.16, 1.01, 'Preliminary', fontsize=13, style='italic', ha='left', va='bottom', transform=ax.transAxes)
    ax.text(1.0, 1.01, f'{year} (13.6 TeV)' if year else '(13.6 TeV)', fontsize=13, ha='right', va='bottom', transform=ax.transAxes)
    # optional descriptive label (e.g. distinguishing target vs reference in
    # a side-by-side comparison), placed above the CMS row so it doesn't clash
    if title:
        ax.text(0.5, 1.10, title, fontsize=11, ha='center', va='bottom', transform=ax.transAxes)

    return fig, ax, im

In [ ]:
def plot_trackmap(run, ls, ax=None, year=None, title=None, **kwargs):
    '''Fetches and plots a single (run, ls) track map. See _plot_trackmap_data().
    Returns (fig, ax, im), or None if no data was found.'''
    res = get_trackmap(run, ls, **kwargs)
    if res is None:
        return None
    data, x_min, x_max, y_min, y_max, entries, dataset = res
    return _plot_trackmap_data(data, x_min, x_max, y_min, y_max, ax=ax, year=year, run=run, title=title)

In [ ]:
def plot_trackmap_sum(run, ls_min=None, ls_max=None, ax=None, year=None, title=None, **kwargs):
    '''Fetches, sums and plots a track map over lumisections [ls_min, ls_max]
    in a single run (both None = the full run). See get_trackmap_sum().
    Returns (fig, ax, im), or None if no data was found.'''
    res = get_trackmap_sum(run, ls_min=ls_min, ls_max=ls_max, **kwargs)
    if res is None:
        return None
    data, x_min, x_max, y_min, y_max, entries, n_ls, dataset = res
    return _plot_trackmap_data(data, x_min, x_max, y_min, y_max, ax=ax, year=year, run=run, title=title)

In [ ]:
def _compute_ratio_significance(target_res, ref_res):
    '''Shared math between compare_trackmaps() and plot-free sweeps: normalizes
    the target/reference maps to a common scale (see normalize_top_half()) and
    returns (ratio, significance), where significance is (norm_target -
    norm_ref) / sigma, with sigma the Poisson (sqrt of bin content)
    uncertainty on each map, scaled by the same normalization factor and
    combined in quadrature.'''
    data, rdata = target_res[0], ref_res[0]
    norm_target, target_scale = normalize_top_half(data)
    norm_ref, ref_scale = normalize_top_half(rdata)
    ratio = norm_target / norm_ref
    unc_target = np.sqrt(data) / target_scale
    unc_ref = np.sqrt(rdata) / ref_scale
    unc_diff = np.sqrt(unc_target**2 + unc_ref**2)
    significance = np.divide(
        norm_target - norm_ref, unc_diff,
        out=np.zeros_like(unc_diff), where=unc_diff > 0,
    )
    return ratio, significance

In [ ]:
def compare_trackmaps(target_run, target_ls, ref_run=None, ref_ls_min=None, ref_ls_max=None,
                       ratio_vmin=0, ratio_vmax=2, sig_vmin=-10, sig_vmax=10,
                       target_res=None, ref_res=None, me=ME, dataset_regex=DATASET_REGEX):
    '''Plots a single target lumisection, a reference built by summing
    lumisections [ref_ls_min, ref_ls_max] of ref_run (defaults to target_run;
    both None = that run's full statistics), a third panel with the ratio of
    the two after each has been normalized to a common scale (see
    normalize_top_half()) -- so the ratio is meaningful despite the target
    (1 LS) and reference (many summed LS) being at very different absolute
    scales -- and a fourth panel with the significance of the deviation (see
    _compute_ratio_significance()). target_res/ref_res can each be passed a
    precomputed get_trackmap()/get_trackmap_sum() result to reuse instead of
    refetching it (e.g. when comparing several target lumisections against
    the same full-run reference). Returns (fig, axs, ratio, significance),
    where axs has 4 (possibly-None-content) axes, and ratio/significance are
    the 2D arrays underlying the last two panels (None if target_res or
    ref_res is unavailable).'''
    if ref_run is None:
        ref_run = target_run

    if target_res is None:
        target_res = get_trackmap(target_run, target_ls, me=me, dataset_regex=dataset_regex)
    if ref_res is None:
        ref_res = get_trackmap_sum(ref_run, ls_min=ref_ls_min, ls_max=ref_ls_max, me=me, dataset_regex=dataset_regex)

    fig, axs = plt.subplots(1, 4, figsize=(25, 6))
    ratio, significance = None, None

    if target_res is not None:
        data, x_min, x_max, y_min, y_max, entries, dataset = target_res
        _plot_trackmap_data(data, x_min, x_max, y_min, y_max, ax=axs[0], run=target_run,
                             title=f'Target: Run {target_run}, LS {target_ls}')

    if ref_res is not None:
        rdata, rx_min, rx_max, ry_min, ry_max, rentries, n_ls, rdataset = ref_res
        if ref_ls_min is None and ref_ls_max is None:
            ref_title = f'Reference: Run {ref_run}, full run'
        else:
            ref_title = f'Reference: Run {ref_run}, LS {ref_ls_min}-{ref_ls_max}'
        _plot_trackmap_data(rdata, rx_min, rx_max, ry_min, ry_max, ax=axs[1], run=ref_run, title=ref_title)

    if target_res is not None and ref_res is not None:
        x_min, x_max, y_min, y_max = target_res[1], target_res[2], target_res[3], target_res[4]
        ratio, significance = _compute_ratio_significance(target_res, ref_res)
        _plot_trackmap_data(ratio, x_min, x_max, y_min, y_max, ax=axs[2], run=target_run,
                             title='Ratio',
                             cmap='coolwarm', vmin=ratio_vmin, vmax=ratio_vmax,
                             cbar_label='Ratio (norm. target / norm. reference)')

        _plot_trackmap_data(significance, x_min, x_max, y_min, y_max, ax=axs[3], run=target_run,
                             title='Significance',
                             cmap='coolwarm', vmin=sig_vmin, vmax=sig_vmax,
                             cbar_label='(target - reference) / stat. unc.')

    fig.tight_layout()
    return fig, axs, ratio, significance

In [ ]:
def save_trackmap(run, ls, plot_dir=PLOT_DIR, **kwargs):
    '''Plots a single (run, ls) track map and saves it to
    <plot_dir>/trackmap_<run>_<ls>.png. Returns the saved path, or None if
    no data was found.'''
    res = plot_trackmap(run, ls, **kwargs)
    if res is None:
        return None
    fig, ax, im = res
    os.makedirs(plot_dir, exist_ok=True)
    path = os.path.join(plot_dir, f'trackmap_{run}_{ls}.png')
    fig.savefig(path, dpi=150, bbox_inches='tight')
    print(f'Saved to {os.path.abspath(path)}')
    return path

In [ ]:
def save_compare_trackmaps(target_run, target_ls, ref_run=None, ref_ls_min=None, ref_ls_max=None, plot_dir=PLOT_DIR, **kwargs):
    '''Plots and saves a target-vs-reference-vs-ratio-vs-significance
    comparison (see compare_trackmaps()) to
    <plot_dir>/trackmap_compare_<target_run>_<target_ls>_vs_<ref_tag>.png.
    Returns (path, ratio, significance) -- the saved path plus the 2D arrays
    underlying the ratio/significance panels (see compare_trackmaps()).'''
    fig, axs, ratio, significance = compare_trackmaps(target_run, target_ls, ref_run=ref_run, ref_ls_min=ref_ls_min, ref_ls_max=ref_ls_max, **kwargs)
    os.makedirs(plot_dir, exist_ok=True)
    ref_run_tag = ref_run if ref_run is not None else target_run
    ref_tag = 'fullrun' if ref_ls_min is None and ref_ls_max is None else f'{ref_ls_min}-{ref_ls_max}'
    path = os.path.join(plot_dir, f'trackmap_compare_{target_run}_{target_ls}_vs_{ref_run_tag}_{ref_tag}.png')
    fig.savefig(path, dpi=150, bbox_inches='tight')
    print(f'Saved to {os.path.abspath(path)}')
    return path, ratio, significance

In [ ]:
# plot each configured (run, ls) in its own figure
for run, ls in RUNS_LS:
    plot_trackmap(run, ls)
    plt.show()


In [ ]:
# alternative: all configured (run, ls) as a grid of subplots
ncols = 3
nrows = -(-len(RUNS_LS) // ncols)  # ceil division
fig, axs = plt.subplots(nrows, ncols, figsize=(5*ncols, 5*nrows), squeeze=False)
for idx, (run, ls) in enumerate(RUNS_LS):
    ax = axs[idx // ncols][idx % ncols]
    plot_trackmap(run, ls, ax=ax)
for idx in range(len(RUNS_LS), nrows*ncols):
    axs[idx // ncols][idx % ncols].axis('off')
fig.tight_layout()
plt.show()

In [ ]:
# run all configured target-vs-reference-vs-ratio comparisons
for target_run, target_ls, ref_run, ref_ls_min, ref_ls_max in COMPARISONS:
    save_compare_trackmaps(target_run, target_ls, ref_run=ref_run, ref_ls_min=ref_ls_min, ref_ls_max=ref_ls_max)
    plt.show()

## Automated sweep over flagged anomalies

Instead of manually entering lumisections, read the flagged-anomaly lists and, for each, pick a single representative lumisection roughly in the middle of its flagged window as the target, compared against the full run it belongs to. Two input sources feed a common anomaly format (see `load_anomalies()`):
- barrel (BPix): `nmf/anomalies.csv`, as produced by `nmf/summarize_anomalies.py` -- keeping only the multi-layer anomalies (ignoring single-layer ones).
- endcap (FPix): `nmf/anomalies_alessandro/Pixel_anomalies_11Dic25.xlsx` -- keeping only the multi-disk anomalies.

Everything downstream (reference caching, the sweep itself, the ratio/significance distributions) treats both sources identically; `INCLUDE_BPIX`/`INCLUDE_FPIX` below control whether either or both are included.

In [ ]:
import csv
import openpyxl

ANOMALIES_CSV = '../nmf/anomalies.csv'
ANOMALY_TYPE = 'multi-layer'

ANOMALIES_XLSX = '../nmf/anomalies_alessandro/Pixel_anomalies_20260825.xlsx'
DISK_ANOMALY_TYPE = 'Multi-Disk'
LAYER_ANOMALY_TYPE = 'Multi-Layer'

LAYER_SOURCE = 'xlsx'   # 'xlsx' (default, matches FPix) or 'csv' (cross-check)

INCLUDE_BPIX = True
INCLUDE_FPIX = True

# anomalies outside [MIN_YEAR, MAX_YEAR] are dropped in load_anomalies().
# MIN_YEAR: there's no DIALS ZeroBias track data available for pre-2024
# runs, so they can never be processed anyway (see the
# run_distributions_cache coverage check). MAX_YEAR: this study is scoped
# to 2024+2025 for now -- later years (e.g. 2026) may already have their
# reference runs cached (harmless/reusable ahead of time), but shouldn't be
# included in the actual analysis yet.
MIN_YEAR = 2024
MAX_YEAR = 2025

ANOMALY_PLOT_DIR = 'output_20260825_etaclip/anomalies'

In [ ]:
# Per-anomaly overrides, for cases where the default choice (reference =
# the anomaly's own run, target = the midpoint LS of its flagged window)
# gives a misleading comparison -- e.g. the anomaly spans (near) the entire
# run, so the run's own statistics are themselves biased by the very
# anomaly being characterized; or the flagged window overlaps a separate
# permanent/masked issue, diluting the visible effect at the midpoint LS.
# To be extended as more such cases are found.

# reference-run overrides, applied to every anomaly in the given run:
# {anomaly's run: run to use as reference instead}
REFERENCE_RUN_OVERRIDES = {
    380237: 380444,
    380238: 380444,
    384032: 383996,   # anomaly lasts the whole run
    384033: 383996,   # anomaly lasts the whole run
    391673: 391670,   # anomaly lasts the whole run
    391674: 391670,   # (391673-391679: same issue)
    391675: 391670,
    391676: 391670,
    391677: 391670,
    391678: 391670,
    391679: 391670,
    391949: 392221,   # FPix ratio > 0.8 under self-reference -- biased reference run
    391950: 392221,   # (391949-391952: same issue)
    391951: 392221,
    391952: 392221,
    395428: 395426,   # anomaly lasts the whole run
    395430: 395426,   # anomaly lasts the whole run
    398012: 397793,   # anomaly lasts the whole run (398012-398390)
    398013: 397793,
    398014: 397793,
    398015: 397793,
    398390: 397793,
    398648: 398902,   # anomaly lasts the whole run
}

# per-anomaly reference-run overrides, for when only one of several
# anomalies sharing a run needs a different reference (so a run-level
# override above would be too broad) -- takes precedence over
# REFERENCE_RUN_OVERRIDES. {(run, target_ls): run to use as reference}
REFERENCE_RUN_OVERRIDES_BY_ANOMALY = {
    # run 398647 has two anomalies (target_ls 1124 and 1337); only the 1337
    # one suffered from an edge-bin artifact dominating the significance
    # metric under self-reference, and 398396 fixes that specifically for
    # 1337 -- but makes 1124 worse (introduces a broad, non-localized
    # apparent deficit, a sign of a general conditions mismatch), so it's
    # kept off the run-level override and only applied here.
    (398647, 1337): 398396,
    # run 397191 has two anomalies (target_ls 27 and 52); only the 27 one
    # (FPix ratio > 0.8 under self-reference -- biased reference run) needs
    # 397190 as reference -- the 52 one was separately tuned (see
    # TARGET_LS_OVERRIDES) and keeps self-reference, so a run-level override
    # would be too broad here too.
    (397191, 27): 397190,
}

# target-LS overrides: {(run, default target_ls): target_ls to use instead}
TARGET_LS_OVERRIDES = {
    (385888, 202): 203,   # effect is more pronounced at LS 203
    (392250, 634): 633,   # match the LS used in the manual example plot
    (396871, 745): 743,   # match the LS used in the manual example plot
    (397191, 51): 52,     # effect is more pronounced at LS 52
}


def resolve_reference_run(anomaly, reference_run_overrides=REFERENCE_RUN_OVERRIDES,
                           reference_run_overrides_by_anomaly=REFERENCE_RUN_OVERRIDES_BY_ANOMALY):
    '''Resolves which run to use as reference for a given anomaly dict (see
    load_anomalies()): a per-anomaly override (keyed by (run, target_ls)) if
    present, else a per-run override (keyed by run), else the anomaly's own
    run.'''
    key = (anomaly['run'], anomaly['target_ls'])
    if key in reference_run_overrides_by_anomaly:
        return reference_run_overrides_by_anomaly[key]
    return reference_run_overrides.get(anomaly['run'], anomaly['run'])

In [ ]:
def load_layer_anomalies_csv(path=ANOMALIES_CSV, anomaly_type=ANOMALY_TYPE, target_ls_overrides=TARGET_LS_OVERRIDES):
    '''Reads the barrel (BPix) anomalies CSV (as produced by
    nmf/summarize_anomalies.py) and returns a list of dicts (detector='BPix',
    run, first_ls, last_ls, n_ls, layers, n_layers, target_ls), filtered to
    the given type ('multi-layer' or 'single-layer'). target_ls is a single
    lumisection roughly in the middle of the anomaly's flagged window, used
    as a representative target for compare_trackmaps() -- unless overridden
    via target_ls_overrides (keyed by (run, default target_ls), see
    TARGET_LS_OVERRIDES). n_layers is the number of distinct layers in the
    semicolon-separated `layers` field -- for anomaly_type 'multi-layer',
    n_layers == 1 identifies anomalies that are topologically "multi-layer"
    but where the layers field lists only one layer, typically a
    single-layer defect overlapping a separate permanent/masked issue
    rather than a genuine multi-layer effect (see REFERENCE_RUN_OVERRIDES).
    Kept for cross-checking against load_layer_anomalies_xlsx() -- see
    LAYER_SOURCE for which one load_anomalies() actually uses.'''
    anomalies = []
    with open(path) as f:
        for row in csv.DictReader(f):
            if row['type'] != anomaly_type:
                continue
            run = int(row['run'])
            first_ls = int(row['first lumisection'])
            last_ls = int(row['last lumisection'])
            layers = row['layers']
            target_ls = (first_ls + last_ls) // 2
            target_ls = target_ls_overrides.get((run, target_ls), target_ls)
            anomalies.append({
                'detector': 'BPix',
                'run': run,
                'first_ls': first_ls,
                'last_ls': last_ls,
                'n_ls': int(row['number of lumisections']),
                'layers': layers,
                'n_layers': len(layers.split(';')),
                'target_ls': target_ls,
            })
    return anomalies


_DISK_RE = re.compile(r'[+-]\s*\d+')
_YEAR_RE = re.compile(r'\d+')


def _parse_disks(disk_str):
    '''Parses a "Layer/Disk" field from the endcap anomalies spreadsheet
    (e.g. "Disk +1+2+3", "Disk -1-2-3", "Disk -1:-2", "Disk -1 -2 -3") into
    a list of signed disk labels, e.g. ["+1", "+2", "+3"]. Formatting in the
    source sheet is inconsistent (+/-/: separators, with or without spaces),
    so this just extracts every sign-digit run rather than relying on a
    specific separator.'''
    return [m.replace(' ', '') for m in _DISK_RE.findall(disk_str or '')]


def _parse_bpix_layers(layer_str):
    '''Parses a "Layer/Disk" field from the barrel rows of the endcap/barrel
    anomalies spreadsheet (e.g. "Layer 1-2", "Layer 1-2-3", "Layer 4",
    "All") into a list of layer labels, e.g. ["1", "2"]. "All" means all
    four layers. Unlike _parse_disks(), the '-' here separates a literal
    list of layer numbers, not a sign -- e.g. "Layer 1-4" means layers 1
    and 4 only (not "1 through 4"; that case is spelled "All" in the
    sheet).'''
    s = (layer_str or '').strip()
    if s == 'All':
        return ['1', '2', '3', '4']
    if s.startswith('Layer '):
        s = s[len('Layer '):]
    return [tok for tok in s.split('-') if tok]


def _load_year_by_run(path=ANOMALIES_XLSX):
    '''Reads the spreadsheet's Year column into a {run: year} map -- year
    is parsed as the leading integer, so e.g. "2023-HI" reads as 2023. Used
    by load_anomalies() to drop pre-MIN_YEAR anomalies regardless of source
    (see there); a run not present in the sheet (e.g. a CSV-only run, which
    only ever start at 2024 anyway) is treated as unknown, not pre-MIN_YEAR.'''
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb.active
    year_by_run = {}
    for row in ws.iter_rows(min_row=2, values_only=True):
        run, year = row[0], row[9]
        if run is None or year is None:
            continue
        m = _YEAR_RE.match(str(year))
        if m:
            year_by_run[int(run)] = int(m.group())
    return year_by_run


def _load_xlsx_anomalies(path, anomaly_type, barrel_forward, detector, layer_parser, target_ls_overrides):
    '''Shared implementation behind load_layer_anomalies_xlsx()/
    load_disk_anomalies(): reads the anomalies spreadsheet, keeps rows with
    Type == anomaly_type and Barrel/Forward == barrel_forward, drops rows
    marked 'Acceptable' in the Acceptable column (a manual reviewer call
    that the anomaly's impact is negligible and it shouldn't be treated as
    a real issue), and returns a list of dicts in the common anomaly schema
    (detector, run, first_ls, last_ls, n_ls, layers, n_layers, target_ls) --
    layers holds the raw "Layer/Disk" text, and n_layers is
    len(layer_parser(layers)).'''
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb.active
    anomalies = []
    for row in ws.iter_rows(min_row=2, values_only=True):
        run, first_ls, last_ls, n_ls, bf, layer_disk, acceptable, rtype = row[:8]
        if rtype != anomaly_type or bf != barrel_forward or run is None or first_ls is None or last_ls is None:
            continue
        if acceptable is not None and str(acceptable).strip().lower() == 'acceptable':
            continue
        run, first_ls, last_ls = int(run), int(first_ls), int(last_ls)
        target_ls = (first_ls + last_ls) // 2
        target_ls = target_ls_overrides.get((run, target_ls), target_ls)
        anomalies.append({
            'detector': detector,
            'run': run,
            'first_ls': first_ls,
            'last_ls': last_ls,
            'n_ls': int(n_ls) if n_ls is not None else (last_ls - first_ls + 1),
            'layers': layer_disk,
            'n_layers': len(layer_parser(layer_disk)),
            'target_ls': target_ls,
        })
    return anomalies


def load_disk_anomalies(path=ANOMALIES_XLSX, anomaly_type=DISK_ANOMALY_TYPE, target_ls_overrides=TARGET_LS_OVERRIDES):
    '''Reads the endcap (FPix) anomalies from the spreadsheet (Barrel/
    Forward == 'Forward'). See _load_xlsx_anomalies() for the shared
    machinery, _parse_disks() for how `layers` is parsed into `n_layers`.'''
    return _load_xlsx_anomalies(path, anomaly_type, 'Forward', 'FPix', _parse_disks, target_ls_overrides)


def load_layer_anomalies_xlsx(path=ANOMALIES_XLSX, anomaly_type=LAYER_ANOMALY_TYPE, target_ls_overrides=TARGET_LS_OVERRIDES):
    '''Reads the barrel (BPix) anomalies from the same spreadsheet used for
    FPix (Barrel/Forward == 'Barrel') -- so both detectors come from the
    same manually-curated source, consistently. See _load_xlsx_anomalies()
    for the shared machinery, _parse_bpix_layers() for how `layers` is
    parsed into `n_layers`. This is what load_anomalies() uses for BPix
    when LAYER_SOURCE == 'xlsx' (the default); see load_layer_anomalies_csv()
    for the alternative (our own summarize_anomalies.py output).'''
    return _load_xlsx_anomalies(path, anomaly_type, 'Barrel', 'BPix', _parse_bpix_layers, target_ls_overrides)


def load_anomalies(include_layer=INCLUDE_BPIX, include_disk=INCLUDE_FPIX, target_ls_overrides=TARGET_LS_OVERRIDES,
                    layer_source=LAYER_SOURCE, layer_path=ANOMALIES_CSV, layer_type=ANOMALY_TYPE,
                    disk_path=ANOMALIES_XLSX, disk_type=DISK_ANOMALY_TYPE, min_year=MIN_YEAR, max_year=MAX_YEAR):
    '''Combines the BPix reader (load_layer_anomalies_xlsx() if
    layer_source=='xlsx', the default, else load_layer_anomalies_csv()) and/or
    load_disk_anomalies() (FPix) into a single list of anomaly dicts in a
    common schema -- so downstream code (run_anomaly_sweep(),
    ratio_at_max_significance(), ...) doesn't need to care which detector
    partition or input source an anomaly came from. include_layer/
    include_disk select which detector(s) to include (both by default, per
    INCLUDE_BPIX/INCLUDE_FPIX); either can be run alone. Note: when
    layer_source=='csv', layer_path/layer_type apply to
    load_layer_anomalies_csv(); when =='xlsx', BPix is read from disk_path
    instead (with LAYER_ANOMALY_TYPE), since both detectors share the same
    spreadsheet in that mode. Anomalies outside [min_year, max_year]
    (default MIN_YEAR/MAX_YEAR) are dropped (see _load_year_by_run()) --
    below min_year there's no DIALS data (those runs can never be
    processed anyway), and above max_year the study is deliberately not
    scoped there yet, even if reference runs happen to already be cached.'''
    anomalies = []
    if include_layer:
        if layer_source == 'xlsx':
            anomalies += load_layer_anomalies_xlsx(disk_path, LAYER_ANOMALY_TYPE, target_ls_overrides)
        elif layer_source == 'csv':
            anomalies += load_layer_anomalies_csv(layer_path, layer_type, target_ls_overrides)
        else:
            raise ValueError(f"layer_source must be 'xlsx' or 'csv', got {layer_source!r}")
    if include_disk:
        anomalies += load_disk_anomalies(disk_path, disk_type, target_ls_overrides)
    year_by_run = _load_year_by_run(disk_path)
    anomalies = [a for a in anomalies if min_year <= year_by_run.get(a['run'], min_year) <= max_year]
    return anomalies

### Cache full-run reference track maps to disk

Fetching a full-run summed track map (`get_trackmap_sum`) means paginating through every lumisection of that run, which dominates the sweep's runtime. Since the same ~120 runs get reused across (re-)runs of the sweep, cache each run's summed map to disk once so later steps can load it instead of re-querying DIALS.

In [ ]:
# directory holding one cached full-run summed track map per run -- shared
# across different anomaly-set configurations (e.g. CSV-sourced vs
# xlsx-sourced BPix), since it only depends on the run number, not on which
# anomalies led us to fetch it. Deliberately stored at the *full* eta range
# regardless of CLIP_ETA_RANGE (cache_reference_trackmaps() below fetches
# with clip_eta_range=None) -- so the cache stays reusable even if the eta
# cut changes later; load_cached_trackmap_sum() applies the clip after
# loading instead.
REFERENCE_CACHE_DIR = 'run_distributions_cache'


def cache_reference_trackmaps(runs, cache_dir=REFERENCE_CACHE_DIR, overwrite=False,
                               me=ME, dataset_regex=DATASET_REGEX):
    '''Fetches the full-run summed track map (full eta range -- see
    REFERENCE_CACHE_DIR) for each run in `runs` and caches it to
    <cache_dir>/run_<run>.npz (one file per run). If overwrite=False
    (default), a run whose cache file already exists is skipped entirely --
    not re-fetched from DIALS. If overwrite=True, every run is re-fetched
    and its cache file replaced. Runs with no data are not cached (and are
    retried on a later call). Returns (cached, skipped, missing):
    dicts/lists of run -> path for freshly-fetched and already-present
    runs, and a list of runs for which no data was found.'''
    os.makedirs(cache_dir, exist_ok=True)
    fetched, skipped, missing = {}, {}, []
    for run in runs:
        path = os.path.join(cache_dir, f'run_{run}.npz')
        if not overwrite and os.path.exists(path):
            skipped[run] = path
            continue
        res = get_trackmap_sum(run, me=me, dataset_regex=dataset_regex, clip_eta_range=None)
        if res is None:
            missing.append(run)
            continue
        data, x_min, x_max, y_min, y_max, entries, n_ls, dataset = res
        np.savez(path, data=data, x_min=x_min, x_max=x_max, y_min=y_min, y_max=y_max,
                  entries=entries, n_ls=n_ls, dataset=dataset)
        fetched[run] = path
    return fetched, skipped, missing


def load_cached_trackmap_sum(run, cache_dir=REFERENCE_CACHE_DIR, clip_eta_range=CLIP_ETA_RANGE):
    '''Loads a full-run summed track map previously cached by
    cache_reference_trackmaps() (always stored at full eta range -- see
    REFERENCE_CACHE_DIR), then restricts it to clip_eta_range (see
    _clip_eta(); defaults to CLIP_ETA_RANGE, pass None to get the cached
    file's full range) -- the cache file itself is never rewritten by this.
    Returns (data, x_min, x_max, y_min, y_max, entries, n_ls, dataset) --
    the same shape as get_trackmap_sum() -- or None if this run isn't
    cached. entries is recomputed from the (possibly clipped) data.'''
    path = os.path.join(cache_dir, f'run_{run}.npz')
    if not os.path.exists(path):
        return None
    with np.load(path) as f:
        data, x_min, x_max, y_min, y_max, n_ls, dataset = (
            f['data'], float(f['x_min']), float(f['x_max']), float(f['y_min']), float(f['y_max']),
            int(f['n_ls']), str(f['dataset']),
        )
    data, x_min, x_max, y_min, y_max = _clip_eta(data, x_min, x_max, y_min, y_max, clip_eta_range)
    return data, x_min, x_max, y_min, y_max, int(data.sum()), n_ls, dataset

In [ ]:
# cache the full-run reference track map for every run appearing in the
# anomalies (BPix and/or FPix, per INCLUDE_BPIX/INCLUDE_FPIX), plus any runs
# used as an explicit reference-run override (see REFERENCE_RUN_OVERRIDES,
# REFERENCE_RUN_OVERRIDES_BY_ANOMALY) -- safe to re-run: already-cached
# runs are skipped
anomaly_runs = sorted(
    {a['run'] for a in load_anomalies()}
    | set(REFERENCE_RUN_OVERRIDES.values())
    | set(REFERENCE_RUN_OVERRIDES_BY_ANOMALY.values())
)
fetched, skipped, missing = cache_reference_trackmaps(anomaly_runs, overwrite=False)
print(f'{len(anomaly_runs)} runs total: {len(fetched)} freshly cached, '
      f'{len(skipped)} already present (skipped), {len(missing)} with no data.')
if missing:
    print('Runs with no data:', missing)

In [ ]:
def run_anomaly_sweep(anomalies, plot_dir=ANOMALY_PLOT_DIR, reference_cache_dir=REFERENCE_CACHE_DIR,
                       reference_run_overrides=REFERENCE_RUN_OVERRIDES,
                       reference_run_overrides_by_anomaly=REFERENCE_RUN_OVERRIDES_BY_ANOMALY,
                       make_plots=True, **kwargs):
    '''Computes (and, if make_plots, plots+saves) a target-vs-reference
    comparison for each anomaly (see load_anomalies()), comparing its
    representative target_ls against the full run it belongs to -- unless
    overridden (see resolve_reference_run(), reference_run_overrides,
    reference_run_overrides_by_anomaly), in which case the resolved run is
    used as reference instead. When make_plots, each anomaly's plot is saved
    under <plot_dir>/<bpix|fpix>/ (from the anomaly's own 'detector' field)
    -- so a sweep mixing both detectors still separates their plots, and a
    single-detector sweep still lands in the right subfolder. The full-run
    reference is loaded from the on-disk cache built by
    cache_reference_trackmaps() (see load_cached_trackmap_sum()) instead of
    querying DIALS -- run cache_reference_trackmaps() on the relevant runs
    first (including any override reference runs). Anomalies whose target
    lumisection has no data, whose reference run isn't cached, or whose
    reference run has zero total entries in the (possibly eta-clipped)
    map -- meaning normalize_top_half()'s scale is 0, so ratio (and, absent
    the significance formula's explicit unc_diff > 0 guard, sometimes
    significance too) would be a mathematically undefined 0/0 or x/0 rather
    than a real "total loss" data point -- are skipped (with a printed
    warning) -- ratio/significance/extent stay None for those, so they
    still appear in the output (see summarize_anomaly_result()) rather than
    silently disappearing (or worse, appearing valid with a NaN ratio that
    np.histogram() then silently drops).
    make_plots=False skips figure creation/saving entirely and only computes
    ratio/significance (see _compute_ratio_significance()) -- much faster
    when only a numerical summary (e.g. ratio_at_max_significance()) is
    needed. Returns a list of per-anomaly result dicts (one per input
    anomaly, in order), each the anomaly's fields plus 'ref_run' (the
    resolved reference run), 'path' (saved plot path, or None if skipped or
    make_plots=False), 'ratio'/'significance' (the 2D arrays, or None if
    skipped), and 'extent' (the (x_min, x_max, y_min, y_max) of those
    arrays, or None if skipped).'''
    ref_cache = {}
    results = []
    for a in anomalies:
        run = a['run']
        ref_run = resolve_reference_run(a, reference_run_overrides, reference_run_overrides_by_anomaly)
        if ref_run not in ref_cache:
            ref_cache[ref_run] = load_cached_trackmap_sum(ref_run, cache_dir=reference_cache_dir)
            if ref_cache[ref_run] is None:
                print(f'WARNING: run {ref_run} (reference for run {run}) not found in reference '
                      f'cache ({reference_cache_dir}); run cache_reference_trackmaps() first. Skipping.')
        ref_res = ref_cache[ref_run]

        result = dict(a, ref_run=ref_run, path=None, ratio=None, significance=None, extent=None)
        if ref_res is not None and ref_res[0].sum() == 0:
            print(f'WARNING: skipping anomaly (run {run}, target LS {a["target_ls"]}, '
                  f'layers {a["layers"]}): reference run {ref_run} has zero total entries '
                  f'-- ratio/significance would be undefined, not a real measurement.')
        elif ref_res is not None:
            target_res = get_trackmap(run, a['target_ls'])
            if target_res is None:
                print(f'WARNING: skipping anomaly (run {run}, target LS {a["target_ls"]}, '
                      f'layers {a["layers"]}): no track data for the target lumisection.')
            else:
                if make_plots:
                    anomaly_plot_dir = os.path.join(plot_dir, a['detector'].lower())
                    path, ratio, significance = save_compare_trackmaps(
                        run, a['target_ls'], ref_run=ref_run, plot_dir=anomaly_plot_dir,
                        target_res=target_res, ref_res=ref_res, **kwargs,
                    )
                    result['path'] = path
                else:
                    ratio, significance = _compute_ratio_significance(target_res, ref_res)
                result['ratio'] = ratio
                result['significance'] = significance
                result['extent'] = (target_res[1], target_res[2], target_res[3], target_res[4])
        results.append(result)
    return results

In [ ]:
# n_bins variants the saved histograms are actually built with (see
# run_anomaly_sweep()'s call sites and ratio_at_top_n_significance()) --
# summarize_anomaly_result() precomputes ratio_top<n> for each of these, so
# the saved JSON alone is enough to reproduce every histogram variant
# without rerunning the sweep or refetching from DIALS.
SUMMARY_TOP_N = (1, 4, 8)


def summarize_anomaly_result(result, top_n=SUMMARY_TOP_N):
    '''Reduces one run_anomaly_sweep() result dict to a compact,
    JSON-serializable summary: which (run, target LS) was evaluated against
    which reference run; at the single bin with the largest |significance|,
    the (eta, phi) bin center, the significance there, and the ratio there
    ('significance'/'ratio', the n=1 case); and, for every n in top_n, the
    ratio averaged over the n bins with the largest |significance|
    ('ratio_top<n>', see ratio_at_top_n_significance()) -- these are exactly
    the per-anomaly values save_ratio_histograms() bins into the saved
    distribution plots, so this JSON is sufficient to reproduce every one of
    them (combined/split, multi-/single-zone, any n_bins in top_n) without
    rerunning the sweep. 'valid' is False (with all of the above left None)
    for anomalies run_anomaly_sweep() skipped (no cached reference run, or
    no data for the target lumisection) -- so every input anomaly still
    produces one entry, valid or not.'''
    ratio, significance, extent = result['ratio'], result['significance'], result['extent']
    valid = ratio is not None and significance is not None and extent is not None
    eta = phi = bin_row = bin_col = significance_value = ratio_value = None
    ratio_top = {n: None for n in top_n}
    if valid:
        row, col = np.unravel_index(np.argmax(np.abs(significance)), significance.shape)
        bin_row, bin_col = int(row), int(col)
        significance_value = float(significance[row, col])
        ratio_value = float(ratio[row, col])
        x_min, x_max, y_min, y_max = extent
        n_rows, n_cols = significance.shape
        eta = x_min + (col + 0.5) * (x_max - x_min) / n_cols
        phi = y_min + (row + 0.5) * (y_max - y_min) / n_rows
        for n in top_n:
            ratio_top[n] = float(ratio_at_top_n_significance(ratio, significance, n=n))
    out = {
        'detector': result['detector'],
        'run': result['run'],
        'ref_run': result['ref_run'],
        'first_ls': result['first_ls'],
        'last_ls': result['last_ls'],
        'target_ls': result['target_ls'],
        'layers': result['layers'],
        'n_layers': result['n_layers'],
        'valid': valid,
        'bin_row': bin_row,
        'bin_col': bin_col,
        'eta': eta,
        'phi': phi,
        'significance': significance_value,
        'ratio': ratio_value,
    }
    for n in top_n:
        out[f'ratio_top{n}'] = ratio_top[n]
    return out


def save_anomaly_summary(results, path):
    '''Writes summarize_anomaly_result() for every entry in results (one
    per input anomaly, whether or not it could be processed -- see
    run_anomaly_sweep()) to a JSON file at path, so individual cases can be
    looked up and interpreted, and every saved distribution plot reproduced,
    afterwards without rerunning the sweep.'''
    os.makedirs(os.path.dirname(path), exist_ok=True)
    summary = [summarize_anomaly_result(r) for r in results]
    with open(path, 'w') as f:
        json.dump(summary, f, indent=2)
    n_valid = sum(s['valid'] for s in summary)
    print(f'Saved {len(summary)} anomalies ({n_valid} valid) to {os.path.abspath(path)}')

In [ ]:
def ratio_at_top_n_significance(ratio, significance, n=1):
    '''Averages the ratio value over the n bins with the largest absolute
    significance -- n=1 reduces to picking the single most significant bin
    (the original ratio_at_max_significance() behavior). Generalization to
    test whether averaging over several significant bins (rather than just
    the single most extreme one, which tends to be dominated by
    low-statistics acceptance-edge bins -- see the baseline threshold-scan
    investigation) gives a cleaner signal. Returns the mean ratio over the
    top n bins by |significance|.'''
    flat_sig = np.abs(significance).ravel()
    flat_ratio = ratio.ravel()
    n = min(n, flat_sig.size)
    top_idx = np.argpartition(flat_sig, -n)[-n:]
    return flat_ratio[top_idx].mean()


def ratio_at_max_significance(source, predicate=None, n_bins=1):
    '''For each anomaly in source, gets the ratio averaged over the n_bins
    bins with the largest |significance| (n_bins=1, the default, picks just
    the single most significant bin). source can be either a live
    run_anomaly_sweep() result list (with 'ratio'/'significance' 2D arrays,
    from which ratio_at_top_n_significance() recomputes the value) or a
    summary loaded from a saved anomaly_summary.json (see
    summarize_anomaly_result(), load_anomaly_summary()), which already
    carries the precomputed value as 'ratio_top<n_bins>' -- so this (and
    save_ratio_histograms(), save_stacked_ratio_histogram(), built on it)
    reproduces exactly the same plots from either source, without rerunning
    the sweep. Anomalies with no data (either way) are excluded, as are
    anomalies for which predicate(anomaly) is False, if a predicate is given
    (e.g. to split by category). Returns a 1D array, one entry per matching
    anomaly with data.'''
    key = f'ratio_top{n_bins}'
    values = []
    for r in source:
        if predicate is not None and not predicate(r):
            continue
        if key in r:
            value = r[key]
        else:
            ratio, significance = r['ratio'], r['significance']
            value = None if (ratio is None or significance is None) else ratio_at_top_n_significance(ratio, significance, n=n_bins)
        if value is not None:
            values.append(value)
    return np.array(values)


# per-scope legend wording for the split histogram (see
# plot_ratio_at_max_significance_hist(), save_ratio_histograms()) --
# 'multi-zone'/'single zone' is the detector-agnostic fallback for any
# scope not listed here.
SCOPE_LABELS = {
    'bpix': ('Multi-layer', 'Single layer with masked overlap'),
    'fpix': ('Multi-disk', 'Single disk with masked overlap'),
    'combined': ('Multi-zone', 'Single zone with masked overlap'),
}


def plot_ratio_at_max_significance_hist(ratio_multizone, ratio_single_masked, combine,
                                         multi_label='Multi-zone',
                                         single_label='Single zone with masked overlap',
                                         hist_bins=np.linspace(0, 1.5, 16)):
    '''Plots the ratio_at_max_significance() distribution(s), either combined
    into a single dodgerblue histogram (combine=True), or as two separate
    overlaid histograms -- dodgerblue (multi_label) for genuine multi-zone
    (multi-layer or multi-disk) anomalies, darkorchid (single_label) for
    single-zone anomalies overlapping a masked issue (combine=False), with a
    legend. Under-/overflow is clipped into the first/last bin rather than
    dropped. Returns (fig, ax).'''
    fig, ax = plt.subplots(figsize=(7, 5))
    if combine:
        ratio_combined = np.concatenate([ratio_multizone, ratio_single_masked])
        ax.hist(np.clip(ratio_combined, hist_bins[0], hist_bins[-1]), bins=hist_bins,
                histtype='step', linewidth=2.5, color='dodgerblue')
    else:
        ax.hist(np.clip(ratio_multizone, hist_bins[0], hist_bins[-1]), bins=hist_bins,
                histtype='step', linewidth=2.5, color='dodgerblue',
                label=multi_label)
        ax.hist(np.clip(ratio_single_masked, hist_bins[0], hist_bins[-1]), bins=hist_bins,
                histtype='step', linewidth=2.5, color='darkorchid',
                label=single_label)
        ax.legend(fontsize=13)
    ax.set_xlabel('Ratio at largest abs. significance', fontsize=14)
    ax.set_ylabel('Number of anomalies', fontsize=14)
    ax.grid(True, alpha=0.3)
    return fig, ax


def save_ratio_histograms(sweep_results, scope, untuned=False, n_bins=1, plot_dir=ANOMALY_PLOT_DIR):
    '''Computes the ratio_at_max_significance() distributions (with n_bins,
    see ratio_at_top_n_significance()) -- split into genuine multi-zone
    (n_layers > 1) vs single-zone-with-masked-overlap (n_layers == 1) -- for
    the given sweep_results, and saves both the combined and split
    histograms (see plot_ratio_at_max_significance_hist(), using the legend
    wording from SCOPE_LABELS[scope]) directly to <plot_dir> (not a
    per-scope subfolder -- plot_dir/bpix and plot_dir/fpix are reserved for
    the individual per-anomaly plots, see run_anomaly_sweep()) as
    ratio_at_max_significance_hist_<scope>[_untuned]_{combined,split}[_top<n_bins>bins].png
    (the _top<n_bins>bins suffix is omitted for the default n_bins=1).'''
    ratio_multizone = ratio_at_max_significance(sweep_results, predicate=lambda r: r['n_layers'] > 1, n_bins=n_bins)
    ratio_single_masked = ratio_at_max_significance(sweep_results, predicate=lambda r: r['n_layers'] == 1, n_bins=n_bins)
    multi_label, single_label = SCOPE_LABELS.get(scope, ('Multi-zone', 'Single zone with masked overlap'))

    os.makedirs(plot_dir, exist_ok=True)
    prefix = 'untuned_' if untuned else ''
    suffix = f'_top{n_bins}bins' if n_bins != 1 else ''
    for combine, tag in [(True, 'combined'), (False, 'split')]:
        fig, ax = plot_ratio_at_max_significance_hist(ratio_multizone, ratio_single_masked, combine=combine,
                                                        multi_label=multi_label, single_label=single_label)
        hist_path = os.path.join(plot_dir, f'ratio_at_max_significance_hist_{scope}_{prefix}{tag}{suffix}.png')
        fig.savefig(hist_path, dpi=150, bbox_inches='tight')
        print(f'Saved to {os.path.abspath(hist_path)}')
        plt.show()

In [ ]:
# run the full (tuned) sweep separately for BPix and FPix -- each saves its
# own individual anomaly plots under {ANOMALY_PLOT_DIR}/<bpix|fpix>/ (see
# run_anomaly_sweep()'s per-anomaly detector routing). Anomalies without a
# cached reference run are still passed through run_anomaly_sweep() (it
# skips them internally, with a warning) rather than filtered out beforehand,
# so they show up as invalid (not missing) in the saved JSON summary. The
# "combined" scope below just concatenates the two already-computed sweeps
# (no extra DIALS calls) to build the combined ratio/significance
# distributions -- a combined *plot directory* would be meaningless since
# every individual anomaly plot already belongs to exactly one detector.
sweep_results_by_scope = {}
for scope, include_layer, include_disk in [('bpix', True, False), ('fpix', False, True)]:
    scope_anomalies = load_anomalies(include_layer=include_layer, include_disk=include_disk)
    n_cached = sum(
        os.path.exists(os.path.join(REFERENCE_CACHE_DIR, f"run_{resolve_reference_run(a)}.npz"))
        for a in scope_anomalies
    )
    print(f'[{scope}] {n_cached} / {len(scope_anomalies)} anomalies have a cached reference run.')

    results = run_anomaly_sweep(scope_anomalies, make_plots=True)
    n_saved = sum(r['path'] is not None for r in results)
    print(f'[{scope}] Saved {n_saved} / {len(results)} plots to {ANOMALY_PLOT_DIR}/{scope}/')
    sweep_results_by_scope[scope] = results

sweep_results_by_scope['combined'] = sweep_results_by_scope['bpix'] + sweep_results_by_scope['fpix']

for scope, results in sweep_results_by_scope.items():
    save_ratio_histograms(results, scope)

save_anomaly_summary(sweep_results_by_scope['combined'], os.path.join(ANOMALY_PLOT_DIR, 'anomaly_summary.json'))

## Distribution of the ratio at the most significant bin

For each swept anomaly, find the bin with the largest absolute significance (see `compare_trackmaps()`), and look up the ratio (norm. target / norm. reference) at that same bin -- i.e. how large the deviation is, at the single bin the anomaly is most detectable at. Plotting this across all anomalies gives a sense of the typical/worst-case tracking impact. Produced for BPix only, FPix only, and combined (see `save_ratio_histograms()`), saved under `output_test/anomalies/<bpix|fpix|combined>/`.

### Comparison: without the per-anomaly fine-tuning

Same distribution, but ignoring `REFERENCE_RUN_OVERRIDES`/`TARGET_LS_OVERRIDES` -- always using the anomaly's own run as reference and the raw midpoint LS as target -- to see how much the fine-tuning actually changes the picture.

In [ ]:
# same idea, but computing (not plotting -- make_plots=False) with
# target_ls_overrides/reference_run_overrides disabled, i.e. no per-anomaly
# fine-tuning at all -- again separately for BPix and FPix, then combined by
# concatenation, and saved with the same 3 scopes' naming (see
# save_ratio_histograms()).
sweep_results_untuned_by_scope = {}
for scope, include_layer, include_disk in [('bpix', True, False), ('fpix', False, True)]:
    scope_untuned_anomalies = load_anomalies(include_layer=include_layer, include_disk=include_disk, target_ls_overrides={})
    n_cached_untuned = sum(
        os.path.exists(os.path.join(REFERENCE_CACHE_DIR, f"run_{a['run']}.npz"))
        for a in scope_untuned_anomalies
    )
    print(f'[{scope}] {n_cached_untuned} / {len(scope_untuned_anomalies)} anomalies '
          f'have a cached reference run (untuned).')

    results = run_anomaly_sweep(
        scope_untuned_anomalies, make_plots=False,
        reference_run_overrides={}, reference_run_overrides_by_anomaly={},
    )
    n_computed = sum(r['ratio'] is not None for r in results)
    print(f'[{scope}] Computed ratio/significance for {n_computed} / {len(results)} anomalies (untuned).')
    sweep_results_untuned_by_scope[scope] = results

sweep_results_untuned_by_scope['combined'] = sweep_results_untuned_by_scope['bpix'] + sweep_results_untuned_by_scope['fpix']

for scope, results in sweep_results_untuned_by_scope.items():
    save_ratio_histograms(results, scope, untuned=True)

save_anomaly_summary(sweep_results_untuned_by_scope['combined'], os.path.join(ANOMALY_PLOT_DIR, 'anomaly_summary_untuned.json'))

In [ ]:
# same anomaly distributions, but averaging over the 4 (then 8) most
# significant bins instead of just the single most significant one (see
# ratio_at_top_n_significance()) -- no new DIALS calls or individual plots
# needed, since ratio/significance are already stored per-anomaly in
# sweep_results_by_scope / sweep_results_untuned_by_scope.
for n_bins in (4, 8):
    for scope, results in sweep_results_by_scope.items():
        save_ratio_histograms(results, scope, n_bins=n_bins)
    for scope, results in sweep_results_untuned_by_scope.items():
        save_ratio_histograms(results, scope, untuned=True, n_bins=n_bins)

## Reproducing the distributions from the saved JSON summaries

The per-anomaly JSON files (`anomaly_summary.json` / `anomaly_summary_untuned.json`, see
`save_anomaly_summary()`) already carry, for every anomaly, the `ratio_top<n>` values
`save_ratio_histograms()` bins into each saved plot -- so the plots above can be regenerated
straight from those files, without rerunning the sweep or touching DIALS. This also doubles as
a check: every plot's file hash is recorded before regenerating it from JSON, and compared
against the hash after, to confirm the reproduction is pixel-for-pixel identical to the
original.

In [ ]:
import hashlib


def _file_hash(path):
    with open(path, 'rb') as f:
        return hashlib.sha256(f.read()).hexdigest()


def load_anomaly_summary(path):
    '''Loads a JSON summary saved by save_anomaly_summary() back into a
    plain list of per-anomaly dicts (see summarize_anomaly_result()).
    Passing this list anywhere a sweep_results list is otherwise expected
    (ratio_at_max_significance(), save_ratio_histograms(),
    save_stacked_ratio_histogram()) reproduces exactly the same plots
    without rerunning the sweep, since it already carries the precomputed
    ratio_top<n> values those functions read.'''
    with open(path) as f:
        return json.load(f)


# {untuned: {scope: [anomaly dicts]}} -- mirrors sweep_results_by_scope /
# sweep_results_untuned_by_scope's structure, but loaded from disk instead
# of kept from the live sweep
summary_by_source = {
    False: load_anomaly_summary(os.path.join(ANOMALY_PLOT_DIR, 'anomaly_summary.json')),
    True: load_anomaly_summary(os.path.join(ANOMALY_PLOT_DIR, 'anomaly_summary_untuned.json')),
}
summary_by_scope = {
    untuned: {
        'bpix': [a for a in summary if a['detector'] == 'BPix'],
        'fpix': [a for a in summary if a['detector'] == 'FPix'],
        'combined': summary,
    }
    for untuned, summary in summary_by_source.items()
}

# snapshot every existing distribution plot's hash before regenerating it
# from the JSON summaries, so the reproduction can be verified byte-for-byte
# -- saved directly under ANOMALY_PLOT_DIR (not a per-scope subfolder, see
# save_ratio_histograms())
hist_paths = [
    os.path.join(ANOMALY_PLOT_DIR, f'ratio_at_max_significance_hist_{scope}_{"untuned_" if untuned else ""}{tag}{"" if n_bins == 1 else f"_top{n_bins}bins"}.png')
    for untuned in (False, True)
    for scope in ('bpix', 'fpix', 'combined')
    for n_bins in (1, 4, 8)
    for tag in ('combined', 'split')
]
before_hashes = {p: _file_hash(p) for p in hist_paths}

for untuned in (False, True):
    for scope in ('bpix', 'fpix', 'combined'):
        for n_bins in (1, 4, 8):
            save_ratio_histograms(summary_by_scope[untuned][scope], scope, untuned=untuned, n_bins=n_bins)

after_hashes = {p: _file_hash(p) for p in hist_paths}
mismatches = [p for p in hist_paths if before_hashes[p] != after_hashes[p]]
print(f'{len(hist_paths)} distribution plots reproduced from the saved JSON summaries; {len(mismatches)} differ from the originals.')
if mismatches:
    for p in mismatches:
        print('  MISMATCH:', p)
else:
    print('All plots are pixel-identical to the ones produced by the live sweep.')

### Extension: stacked by number of layers/disks

The split above only distinguishes genuine multi-zone anomalies (any `n_layers > 1`) from the
single-zone-with-masked-overlap case (`n_layers == 1`), lumping every multi-zone anomaly
together regardless of how many layers/disks it actually spans. This stacks a separate filled
histogram layer per exact `n_layers` value instead -- so e.g. a 2-layer and a 4-layer BPix
anomaly show up in different, distinguishable layers of the stack rather than being merged
into one "multi-zone" bucket -- while keeping the single-zone-masked case as its own bottom
layer, same as before.

In [ ]:
# noun used in the stacked-histogram legend labels, per scope
STACK_NOUN = {'bpix': 'layer', 'fpix': 'disk', 'combined': 'zone'}


def save_stacked_ratio_histogram(source, scope, untuned=False, n_bins=1, plot_dir=ANOMALY_PLOT_DIR,
                                  hist_bins=np.linspace(0, 1.5, 16)):
    '''Extension of save_ratio_histograms(): instead of the binary
    multi-zone vs single-zone-with-masked-overlap split, stacks a separate
    filled histogram layer per exact n_layers value present in source (how
    many layers/disks were flagged for that anomaly) -- n_layers == 1
    (single layer/disk overlapping a masked region) stays its own bottom
    layer (darkorchid, as in save_ratio_histograms()'s split view), and
    every n_layers > 1 value present gets its own stacked layer (colored
    from a sequential colormap, in increasing n_layers order) instead of
    being lumped into one "multi-zone" group. source can be a live
    sweep_results list or a JSON-loaded summary (see
    ratio_at_max_significance()). Saved directly to <plot_dir> (not a
    per-scope subfolder -- see save_ratio_histograms()) as
    ratio_at_max_significance_hist_<scope>_stacked[_untuned][_top<n_bins>bins].png.'''
    noun = STACK_NOUN.get(scope, 'zone')
    n_layers_values = sorted({r['n_layers'] for r in source if r.get('n_layers') is not None})

    groups = []
    for nl in n_layers_values:
        values = ratio_at_max_significance(source, predicate=lambda r, nl=nl: r['n_layers'] == nl, n_bins=n_bins)
        if len(values) == 0:
            continue
        label = f'Single {noun} (masked overlap)' if nl == 1 else f'{nl} {noun}s'
        groups.append([nl, values, label, None])

    multi_colors = plt.cm.viridis(np.linspace(0.85, 0.15, sum(1 for nl, *_ in groups if nl > 1)))
    ci = 0
    for g in groups:
        if g[0] == 1:
            g[3] = 'darkorchid'
        else:
            g[3] = multi_colors[ci]
            ci += 1

    fig, ax = plt.subplots(figsize=(7, 5))
    clipped = [np.clip(values, hist_bins[0], hist_bins[-1]) for _, values, _, _ in groups]
    labels = [f'{label} (N={len(values)})' for _, values, label, _ in groups]
    colors = [color for _, _, _, color in groups]
    ax.hist(clipped, bins=hist_bins, stacked=True, histtype='stepfilled',
            color=colors, label=labels, edgecolor='black', linewidth=0.6, alpha=0.9)
    ax.set_xlabel('Ratio at largest abs. significance', fontsize=14)
    ax.set_ylabel('Number of anomalies', fontsize=14)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=11)

    os.makedirs(plot_dir, exist_ok=True)
    prefix = 'untuned_' if untuned else ''
    suffix = f'_top{n_bins}bins' if n_bins != 1 else ''
    hist_path = os.path.join(plot_dir, f'ratio_at_max_significance_hist_{scope}_{prefix}stacked{suffix}.png')
    fig.savefig(hist_path, dpi=150, bbox_inches='tight')
    print(f'Saved to {os.path.abspath(hist_path)}')
    plt.show()
    return hist_path

In [ ]:
# same three (scope x n_bins) x (tuned/untuned) sweep as above, but the
# stacked-by-n_layers view -- sourced from the same JSON summaries
for untuned in (False, True):
    for scope in ('bpix', 'fpix', 'combined'):
        for n_bins in (1, 4, 8):
            save_stacked_ratio_histogram(summary_by_scope[untuned][scope], scope, untuned=untuned, n_bins=n_bins)

## Baseline cross-check: normal lumisections

The distributions above are for anomalous lumisections; as a cross-check, apply the exact same `ratio_at_max_significance` metric to *normal* (non-flagged) lumisections of a run, compared against that run's own cached aggregate -- this should hopefully come out roughly symmetric around 1, with a modest width, unlike the anomaly distributions which are skewed toward deficits. Saved under `output_test/baseline/`. Starting point: all lumisections of run 379470 except the ones flagged (of *any* type, not just multi-disk/-layer) in either input source -- see `flagged_ls_ranges()`.

In [ ]:
# output directory for the baseline cross-check
BASELINE_PLOT_DIR = 'output_test/baseline'

# run to sample "normal" lumisections from for the baseline; edit/extend freely
BASELINE_RUN = 379470


def get_trackmap_all_ls(run, me=ME, dataset_regex=DATASET_REGEX):
    '''Fetches every individual per-LS 2D histogram for a run (not summed --
    see get_trackmap_sum() for the summed version). Returns a dict
    {ls_number: (data, x_min, x_max, y_min, y_max, entries, dataset)},
    empty if nothing was found.'''
    filters = LumisectionHistogram2DFilters(
        me=me,
        dataset__regex=dataset_regex,
        run_number=run,
    )
    result = dials.h2d.list_all(filters, enable_progress=False)
    return {
        r.ls_number: (np.array(r.data), r.x_min, r.x_max, r.y_min, r.y_max, r.entries, r.dataset)
        for r in result.results
    }


def flagged_ls_ranges(run, layer_path=ANOMALIES_CSV, disk_path=ANOMALIES_XLSX):
    '''Returns a list of (first_ls, last_ls) windows flagged for the given
    run in either input source, of *any* type (not just multi-layer/
    multi-disk -- unlike load_anomalies()) -- used to exclude anomalous
    lumisections when sampling a "normal" baseline from a run.'''
    ranges = []
    with open(layer_path) as f:
        for row in csv.DictReader(f):
            if int(row['run']) == run:
                ranges.append((int(row['first lumisection']), int(row['last lumisection'])))
    wb = openpyxl.load_workbook(disk_path, data_only=True)
    ws = wb.active
    for row in ws.iter_rows(min_row=2, values_only=True):
        r_run, first_ls, last_ls = row[0], row[1], row[2]
        if r_run == run and first_ls is not None and last_ls is not None:
            ranges.append((int(first_ls), int(last_ls)))
    return ranges

In [ ]:
def compute_baseline_ratios(run, exclude_ranges=None, n_bins=1, me=ME, dataset_regex=DATASET_REGEX,
                             reference_cache_dir=REFERENCE_CACHE_DIR):
    '''For every lumisection of `run` not covered by exclude_ranges (default:
    flagged_ls_ranges(run), i.e. every flagged window of any type), computes
    the ratio averaged over the n_bins bins with the largest |significance|
    (see ratio_at_top_n_significance(); n_bins=1, the default, reproduces
    the original single-bin ratio_at_max_significance() metric) against
    that run's own cached full-run reference (see load_cached_trackmap_sum())
    -- the same metric used for anomalies, but applied to presumed-"normal"
    lumisections, as a cross-check of what its distribution looks like when
    nothing is actually wrong. Returns (ratios, max_abs_significances) --
    two 1D arrays, one entry per normal LS with usable data, in the same
    order (max_abs_significances is always the single largest |significance|
    in the map, regardless of n_bins, for use in threshold scans; prints
    how many total/excluded/used).'''
    if exclude_ranges is None:
        exclude_ranges = flagged_ls_ranges(run)

    ref_res = load_cached_trackmap_sum(run, cache_dir=reference_cache_dir)
    if ref_res is None:
        print(f'WARNING: run {run} not found in reference cache ({reference_cache_dir}); '
              f'run cache_reference_trackmaps() first.')
        return np.array([]), np.array([])

    all_ls = get_trackmap_all_ls(run, me=me, dataset_regex=dataset_regex)
    ratios = []
    max_abs_sigs = []
    n_excluded = 0
    for ls, target_res in sorted(all_ls.items()):
        if any(lo <= ls <= hi for lo, hi in exclude_ranges):
            n_excluded += 1
            continue
        ratio, significance = _compute_ratio_significance(target_res, ref_res)
        ratios.append(ratio_at_top_n_significance(ratio, significance, n=n_bins))
        max_abs_sigs.append(np.abs(significance).max())

    print(f'run {run}: {len(all_ls)} total LS, {n_excluded} excluded (flagged), '
          f'{len(ratios)} used for the baseline.')
    return np.array(ratios), np.array(max_abs_sigs)


def save_baseline_histogram(ratio_values, run, plot_dir=BASELINE_PLOT_DIR, suffix='',
                             hist_bins=np.linspace(0, 2, 41)):
    '''Plots and saves the baseline ratio distribution (see
    compute_baseline_ratios()) as a single (uncategorized) histogram -- no
    multi-/single-zone split applies here, since these aren't flagged
    anomalies. Centered range (default 0-2, 40 bins -- narrower bins than
    the anomalies' 0-1.5/15 bins, since there's plenty of statistics here
    to support finer binning) rather than the anomalies' skewed-low range,
    since this is expected to scatter around 1 rather than be skewed toward
    deficits. suffix distinguishes variants (e.g. different n_bins) in the
    filename. Saved to
    <plot_dir>/ratio_at_max_significance_hist_baseline_run<run><suffix>.png.'''
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.hist(np.clip(ratio_values, hist_bins[0], hist_bins[-1]), bins=hist_bins,
            histtype='step', linewidth=2.5, color='dodgerblue')
    ax.axvline(1.0, color='gray', linestyle='--', linewidth=1)
    ax.set_xlabel('Ratio at largest abs. significance', fontsize=14)
    ax.set_ylabel('Number of lumisections', fontsize=14)
    ax.grid(True, alpha=0.3)

    os.makedirs(plot_dir, exist_ok=True)
    hist_path = os.path.join(plot_dir, f'ratio_at_max_significance_hist_baseline_run{run}{suffix}.png')
    fig.savefig(hist_path, dpi=150, bbox_inches='tight')
    print(f'Saved to {os.path.abspath(hist_path)}')
    plt.show()
    return hist_path


def save_baseline_threshold_scan(ratio_values, max_abs_sigs, run, thresholds=None, n_thresholds=6,
                                  plot_dir=BASELINE_PLOT_DIR, suffix='', hist_bins=np.linspace(0, 2, 41)):
    '''Overlays the baseline ratio distribution (see compute_baseline_ratios())
    for a scan of minimum |significance| thresholds -- i.e. each curve only
    includes lumisections whose max |significance| is at least that
    threshold, to see whether imposing a minimum significance cleans up the
    distribution. thresholds defaults to n_thresholds values evenly spaced
    from 0 up to the 90th percentile of max_abs_sigs ("as derived from the
    data"). Curves are colored from the 'cool' colormap. suffix distinguishes
    variants (e.g. different n_bins) in the filename. Saves two versions --
    absolute counts, and each curve normalized to unit area (y-axis
    "... (normalized)") -- to
    <plot_dir>/ratio_at_max_significance_hist_baseline_run<run><suffix>_thresholdscan[_normalized].png.
    Returns the two saved paths.'''
    if thresholds is None:
        thresholds = np.linspace(0, np.percentile(max_abs_sigs, 90), n_thresholds)
    colors = plt.cm.cool(np.linspace(0, 1, len(thresholds)))

    os.makedirs(plot_dir, exist_ok=True)
    paths = []
    for normalized, tag, ylabel in [
        (False, 'thresholdscan', 'Number of lumisections'),
        (True, 'thresholdscan_normalized', 'Number of lumisections (normalized)'),
    ]:
        fig, ax = plt.subplots(figsize=(8, 5.5))
        for threshold, color in zip(thresholds, colors):
            selected = ratio_values[max_abs_sigs >= threshold]
            counts, edges = np.histogram(np.clip(selected, hist_bins[0], hist_bins[-1]), bins=hist_bins)
            if normalized and counts.sum() > 0:
                counts = counts / counts.sum()
            ax.stairs(counts, edges, linewidth=2.2, color=color,
                      label=f'|sig| $\\geq$ {threshold:.1f}  (N={len(selected)})')
        ax.axvline(1.0, color='gray', linestyle='--', linewidth=1)
        ax.set_xlabel('Ratio at largest abs. significance', fontsize=14)
        ax.set_ylabel(ylabel, fontsize=14)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=10)

        hist_path = os.path.join(plot_dir, f'ratio_at_max_significance_hist_baseline_run{run}{suffix}_{tag}.png')
        fig.savefig(hist_path, dpi=150, bbox_inches='tight')
        print(f'Saved to {os.path.abspath(hist_path)}')
        plt.show()
        paths.append(hist_path)
    return paths

In [ ]:
# run the baseline cross-check for BASELINE_RUN
baseline_ratios, baseline_max_abs_sigs = compute_baseline_ratios(BASELINE_RUN)
save_baseline_histogram(baseline_ratios, BASELINE_RUN)
save_baseline_threshold_scan(baseline_ratios, baseline_max_abs_sigs, BASELINE_RUN)

# same, but averaging the ratio over the 4 (then 8) most significant bins
# instead of just the single most significant one (see
# ratio_at_top_n_significance())
baseline_ratios_top4, baseline_max_abs_sigs_top4 = compute_baseline_ratios(BASELINE_RUN, n_bins=4)
save_baseline_histogram(baseline_ratios_top4, BASELINE_RUN, suffix='_top4bins')
save_baseline_threshold_scan(baseline_ratios_top4, baseline_max_abs_sigs_top4, BASELINE_RUN, suffix='_top4bins')

baseline_ratios_top8, baseline_max_abs_sigs_top8 = compute_baseline_ratios(BASELINE_RUN, n_bins=8)
save_baseline_histogram(baseline_ratios_top8, BASELINE_RUN, suffix='_top8bins')
save_baseline_threshold_scan(baseline_ratios_top8, baseline_max_abs_sigs_top8, BASELINE_RUN, suffix='_top8bins')